In [1]:
!git clone https://github.com/sinh2206/O_D.git

Cloning into 'O_D'...
remote: Enumerating objects: 9094, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 9094 (delta 31), reused 44 (delta 14), pack-reused 9031 (from 2)
Receiving objects: 100% (9094/9094), 996.85 MiB | 16.03 MiB/s, done.
Resolving deltas: 100% (32/32), done.
Updating files: 100% (9020/9020), done.


In [2]:
%cd O_D
!ls

/content/O_D
LICENSE  object_detection.ipynb  public     requirements.txt  train.py
models	 predict.py		 README.md  results	      utils


In [3]:
%pip install -r requirements.txt

In [4]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 198MB/s]
Device: cuda:0 (Tesla T4, 14.6 GB), AMP: True
DataLoader workers: 2 (max safe: 2), pin_memory: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Epoch 001/040 | train_loss=2.5904 (obj=0.0468, noobj=1.0029, box=0.2530, cls=0.5901, iou=0.5307) | val_loss=1.9619
Saved best checkpoint: models/best.pth (val_loss=1.9619)
Epoch 002/040 | train_loss=1.9557 (obj=0.0355, noobj=0.7665, box=0.1988, cls=0.4007, iou=0.5612) | val_loss=1.7806
Saved best checkpoint: models/best.pth (val_loss=1.7806)
Epoch 003/040 | train_loss=1.7577 (obj=0.0316, noobj=0.6949, box=0.1783, cls=0.3610, iou=0.5795) | val_loss=1.8019
Epoch 004/040 | train_loss=1.6556 (obj=0.0307, noobj=0.6485, box=0.1652, cls=0.3518, iou=0.5901) | val_loss=1.9589
Epoch 005/040 | train_loss=1.5588 (obj=0.0298, noobj=0.6313, bo

In [5]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda:0 (Tesla T4, 14.6 GB)
Predicted images: 1500
Saved JSON: val_predictions.json
Saved preview images: 50 -> results


In [6]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.368375,
  "performance_points": 15,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 21156,
  "micro_precision": 0.058565,
  "micro_recall": 0.613063,
  "per_class": {
    "person": {
      "ap": 0.363075,
      "num_ground_truth": 1074,
      "num_predictions": 8475,
      "true_positives": 651,
      "false_positives": 7824,
      "recall": 0.606145,
      "precision": 0.076814
    },
    "car": {
      "ap": 0.308139,
      "num_ground_truth": 283,
      "num_predictions": 3865,
      "true_positives": 141,
      "false_positives": 3724,
      "recall": 0.498233,
      "precision": 0.036481
    },
    "dog": {
      "ap": 0.493306,
      "num_ground_truth": 206,
      "num_predictions": 1240,
      "true_positives": 148,
      "false_positives": 1092,
      "recall": 0.718447,
      "precision": 0.119355
    },
    "cat": {
      "ap": 0.547504,
      "num_ground_truth": 176,
      "num_predictions": 946,
      "true_positives": 134,
  

In [7]:
!python predict.py \
  --image_dir ./public/train/images \
  --output train_predictions.json \
  --model_path ./models/best.pth

Device: cuda:0 (Tesla T4, 14.6 GB)
Predicted images: 7500
Saved JSON: train_predictions.json
Saved preview images: 50 -> results


In [8]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/train.json \
  --predictions train_predictions.json \
  --output train_score.json

{
  "mAP@0.5": 0.465137,
  "performance_points": 15,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 10642,
  "num_predictions": 103185,
  "micro_precision": 0.069768,
  "micro_recall": 0.676471,
  "per_class": {
    "person": {
      "ap": 0.378963,
      "num_ground_truth": 5829,
      "num_predictions": 41322,
      "true_positives": 3810,
      "false_positives": 37512,
      "recall": 0.653628,
      "precision": 0.092203
    },
    "car": {
      "ap": 0.40854,
      "num_ground_truth": 1339,
      "num_predictions": 17990,
      "true_positives": 803,
      "false_positives": 17187,
      "recall": 0.599701,
      "precision": 0.044636
    },
    "dog": {
      "ap": 0.667124,
      "num_ground_truth": 1028,
      "num_predictions": 6170,
      "true_positives": 887,
      "false_positives": 5283,
      "recall": 0.86284,
      "precision": 0.14376
    },
    "cat": {
      "ap": 0.744362,
      "num_ground_truth": 833,
      "num_predictions": 4510,
      "true_positives": 

In [9]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/content/O_D')
out_zip = Path('/content/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /content/O_D && zip -r /content/train.zip . -x 'public/*' 'public/**'
Created: /content/train.zip


In [10]:
from google.colab import files
files.download("/content/train.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>